In [1]:
# ── koolbox fallback (to prevent import errors elsewhere, if any) ──
import sys, os, glob, subprocess, types
from pathlib import Path

_koolbox_roots = [Path(p) for p in globals().get('KOOLBOX_OFFLINE_ROOTS', ()) if str(p).strip()]
_koolbox_root = next((p for p in _koolbox_roots if p.exists()), None)
if _koolbox_root is None:
    _auto_hits = []
    for _pat in ('/kaggle/input/**/koolbox*.whl', '/kaggle/input/**/koolbox*'):
        _auto_hits.extend(Path(x).parent if Path(x).suffix == '.whl' else Path(x) for x in glob.glob(_pat, recursive=True))
    _koolbox_root = next((p for p in sorted(set(_auto_hits)) if p.exists()), None)

def _wheel_matches_runtime(path):
    name = Path(path).name
    if ' (' in name or not name.endswith('.whl'):
        return False
    parts = name[:-4].split('-')
    if len(parts) < 5:
        return False
    py_tag, abi_tag, _platform_tag = parts[-3], parts[-2], parts[-1]
    runtime_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    if py_tag.startswith('cp') and py_tag != runtime_tag:
        return False
    if abi_tag.startswith('cp') and abi_tag != runtime_tag:
        return False
    return py_tag in {'py2.py3', 'py3', runtime_tag} or py_tag.startswith(runtime_tag)

def _install_or_path_koolbox(root):
    if root is None:
        return False
    print('using koolbox dir:', root)
    whls = [w for w in sorted(root.glob('**/*.whl')) if _wheel_matches_runtime(w)]
    if whls:
        for w in whls:
            print('install', w)
            subprocess.run(['pip', 'install', '--no-deps', str(w)], check=False)
    else:
        sys.path.insert(0, str(root))
        for sub in root.iterdir():
            if sub.is_dir():
                sys.path.insert(0, str(sub))
    return True

def _make_koolbox_fallback_module():
    # (full fallback code from your snippet – included for completeness)
    import numpy as _np
    import joblib as _joblib
    from pathlib import Path as _Path
    from sklearn.base import clone as _clone
    from sklearn.metrics import root_mean_squared_error as _rmse
    from sklearn.model_selection import GroupKFold as _GroupKFold, KFold as _KFold
    def _take(X, idx):
        return X.iloc[idx] if hasattr(X, 'iloc') else X[idx]
    def _score(metric, y_true, y_pred):
        try:
            return float(metric(y_true, y_pred)) if callable(metric) else float(_rmse(y_true, y_pred))
        except Exception:
            return float(_rmse(y_true, y_pred))
    def _drop_fit_keys(kwargs, keys):
        out = dict(kwargs or {})
        for key in keys:
            out.pop(key, None)
        return out
    class Trainer:
        def __init__(self, estimator, task='regression', metric=None, cv=None, cv_args=None,
                     use_early_stopping=False, verbose=False, save=False, save_path=None):
            self.estimator = estimator
            self.task = task
            self.metric = metric or _rmse
            self.cv = cv
            self.cv_args = cv_args or {}
            self.use_early_stopping = bool(use_early_stopping)
            self.verbose = bool(verbose)
            self.save = bool(save)
            self.save_path = save_path
            self.models = []
            self.oof_preds = None
            self.fold_scores = []
            self.overall_score = None
        def _splits(self, X, y):
            groups = self.cv_args.get('groups')
            cv = self.cv
            if cv is None:
                cv = _GroupKFold(n_splits=5) if groups is not None else _KFold(n_splits=5, shuffle=True, random_state=42)
            try:
                return list(cv.split(X, y, groups=groups))
            except TypeError:
                return list(cv.split(X, y))
        def _fit_one(self, estimator, X_tr, y_tr, X_va=None, y_va=None, fit_args=None):
            fit_kwargs = dict(fit_args or {})
            if self.use_early_stopping and X_va is not None and y_va is not None:
                mod = estimator.__class__.__module__.lower()
                name = estimator.__class__.__name__.lower()
                if 'lightgbm' in mod or 'lgbm' in name:
                    fit_kwargs.setdefault('eval_set', [(X_va, y_va)])
                elif 'catboost' in mod or 'catboost' in name:
                    fit_kwargs.setdefault('eval_set', (X_va, y_va))
            try:
                estimator.fit(X_tr, y_tr, **fit_kwargs)
            except TypeError:
                estimator.fit(X_tr, y_tr, **_drop_fit_keys(fit_kwargs, [
                    'callbacks', 'eval_metric', 'eval_set', 'early_stopping_rounds', 'use_best_model', 'verbose'
                ]))
            return estimator
        def fit(self, X, y, fit_args=None):
            y_arr = _np.asarray(y, dtype=float)
            oof = _np.full(len(y_arr), _np.nan, dtype=float)
            self.models = []
            self.fold_scores = []
            for fold, (tr_idx, va_idx) in enumerate(self._splits(X, y_arr), start=1):
                est = _clone(self.estimator)
                X_tr = _take(X, tr_idx); X_va = _take(X, va_idx)
                y_tr = y_arr[tr_idx]; y_va = y_arr[va_idx]
                est = self._fit_one(est, X_tr, y_tr, X_va, y_va, fit_args=fit_args)
                pred = _np.asarray(est.predict(X_va), dtype=float)
                oof[va_idx] = pred
                score = _score(self.metric, y_va, pred)
                self.fold_scores.append(score)
                self.models.append(est)
                if self.verbose:
                    print(f'fallback Trainer fold {fold}: {score:.5f}')
            if not _np.isfinite(oof).all():
                raise RuntimeError('fallback Trainer produced incomplete OOF predictions')
            self.oof_preds = oof
            self.overall_score = _score(self.metric, y_arr, oof)
            if self.save and self.save_path:
                out_dir = _Path(self.save_path)
                out_dir.mkdir(parents=True, exist_ok=True)
                _joblib.dump(self, out_dir / 'trainer.pkl')
            return self
        def predict(self, X):
            if not self.models:
                raise RuntimeError('Trainer has no fitted fold models')
            preds = [_np.asarray(model.predict(X), dtype=float) for model in self.models]
            return _np.mean(preds, axis=0)
    Trainer.__module__ = 'koolbox'
    Trainer.__qualname__ = 'Trainer'
    module = types.ModuleType('koolbox')
    module.Trainer = Trainer
    module.__file__ = '<fallback koolbox Trainer shim>'
    return module

_koolbox_mode = 'fallback'
try:
    _install_or_path_koolbox(_koolbox_root)
    import koolbox as _koolbox_probe
    _koolbox_mode = 'external'
except Exception as _e:
    print('koolbox external unavailable; using fallback Trainer shim:', _e)
    sys.modules['koolbox'] = _make_koolbox_fallback_module()
    import koolbox as _koolbox_probe

print('koolbox mode:', _koolbox_mode, '| module:', getattr(_koolbox_probe, '__file__', '<unknown>'))

# ── End of koolbox setup ──────────────────────────────────────────────────────

using koolbox dir: /kaggle/input/datasets/phongnguyn23021656/koolbox-offline
install /kaggle/input/datasets/phongnguyn23021656/koolbox-offline/alembic-1.17.2-py3-none-any.whl
Processing /kaggle/input/datasets/phongnguyn23021656/koolbox-offline/alembic-1.17.2-py3-none-any.whl
  Attempting uninstall: alembic
    Found existing installation: alembic 1.18.4
    Uninstalling alembic-1.18.4:
      Successfully uninstalled alembic-1.18.4
install /kaggle/input/datasets/phongnguyn23021656/koolbox-offline/colorlog-6.10.1-py3-none-any.whl
Processing /kaggle/input/datasets/phongnguyn23021656/koolbox-offline/colorlog-6.10.1-py3-none-any.whl
colorlog is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
install /kaggle/input/datasets/phongnguyn23021656/koolbox-offline/joblib-1.5.2-py3-none-any.whl
Processing /kaggle/input/datasets/phongnguyn23021656/koolbox-offline/joblib-1.5.2-py3-none-any.whl
  Attempting uninstall: joblib
  

In [2]:
"""
LOCAL BACKTEST HARNESS – FINAL FIXED VERSION
============================================
- Masks only the last HIDE_FRAC of KNOWN (non‑NaN) rows.
- Correctly detects TVT column (not last_known_tvt).
- Builds full 195‑feature matrix in training order.
- Scores anchor + residual (clipped) for pretrained models.
"""

import os, glob, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from koolbox import Trainer
import sys

warnings.filterwarnings('ignore')
T0 = time.time()
def log(msg): print(f"[{time.time()-T0:6.1f}s] {msg}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────
N_BACKTEST_WELLS = 40
HIDE_FRAC = 0.35
MIN_KNOWN_ROWS = 400
FORM_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c): return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError

INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
ARTIFACT_DIR = '/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts'
TRAIN_CSV_PATH = os.path.join(ARTIFACT_DIR, 'data', 'train.csv')
MODEL_PATHS = [
    os.path.join(ARTIFACT_DIR, 'models', 'catboost-1', 'catboostregressor_trainer_20260526193740.pkl'),
    os.path.join(ARTIFACT_DIR, 'models', 'catboost-2', 'catboostregressor_trainer_20260526194838.pkl'),
    os.path.join(ARTIFACT_DIR, 'models', 'lightgbm-1', 'lgbmregressor_trainer_20260526182612.pkl'),
    os.path.join(ARTIFACT_DIR, 'models', 'lightgbm-2', 'lgbmregressor_trainer_20260526190415.pkl'),
    os.path.join(ARTIFACT_DIR, 'models', 'lightgbm-3', 'lgbmregressor_trainer_20260526192806.pkl'),
]

all_wells = sorted(set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
))
rng = np.random.default_rng(0)
backtest_wells = rng.choice(all_wells, size=min(N_BACKTEST_WELLS, len(all_wells)), replace=False)
log(f"Backtesting on {len(backtest_wells)} wells")

# ── Load artifact means for fallback ──────────────────────────────────
ARTIFACT_MEANS = {}
try:
    art = pd.read_csv(TRAIN_CSV_PATH)
    log("target column summary:")
    print(art['target'].describe())
    ARTIFACT_MEANS = art.select_dtypes(include=[np.number]).mean().to_dict()
except Exception as e:
    log(f"Could not load artifact train.csv: {e}")

# ── Ensure koolbox is importable ──────────────────────────────────────
for root in ['/kaggle/usr/lib', '/kaggle/input']:
    if os.path.isdir(root):
        for hit in glob.glob(os.path.join(root, '**', 'koolbox*'), recursive=True):
            parent = hit if os.path.isdir(hit) else os.path.dirname(hit)
            parent = os.path.dirname(parent) if os.path.basename(parent) == 'koolbox' else parent
            if parent not in sys.path:
                sys.path.insert(0, parent)
try:
    import koolbox
    log("koolbox import OK")
except Exception as e:
    log(f"koolbox not importable ({e}) – model loading may fall back to raw pickle.")

# ── Robust feature name extractor ──────────────────────────────────────
def extract_feature_names(model_obj):
    if hasattr(model_obj, 'estimators') and isinstance(model_obj.estimators, list):
        if len(model_obj.estimators) > 0:
            base = model_obj.estimators[0]
            if isinstance(base, list) and len(base) > 0:
                base = base[0]
            return extract_feature_names(base)
    if hasattr(model_obj, 'feature_names_'):
        return list(model_obj.feature_names_)
    if hasattr(model_obj, 'get_feature_names'):
        try:
            return list(model_obj.get_feature_names())
        except:
            pass
    if hasattr(model_obj, 'feature_name_'):
        return list(model_obj.feature_name_)
    if hasattr(model_obj, 'model'):
        return extract_feature_names(model_obj.model)
    if hasattr(model_obj, '_model'):
        return extract_feature_names(model_obj._model)
    return None

# ── Load models ──────────────────────────────────────────────────────────
PRETRAINED = []
for p in MODEL_PATHS:
    loaded = None
    for attempt in [
        lambda: pickle.load(open(p, 'rb')),
        lambda: pickle.load(open(p, 'rb'), fix_imports=True, encoding='latin1'),
    ]:
        try:
            loaded = attempt()
            break
        except Exception as e:
            last_err = e
    if loaded is None:
        try:
            import joblib
            loaded = joblib.load(p)
        except Exception as e:
            last_err = e
    if loaded is not None:
        feats = extract_feature_names(loaded)
        PRETRAINED.append({
            'name': os.path.basename(p).replace('.pkl', ''),
            'model': loaded,
            'features': feats
        })
        log(f"  Loaded {os.path.basename(p)} (features: {len(feats) if feats else 'unknown'})")
    else:
        log(f"  Could not load {os.path.basename(p)}: {last_err}")
log(f"Pretrained models available: {len(PRETRAINED)}")

# ── Global feature list ──────────────────────────────────────────────────
ALL_FEATURES = None
for entry in PRETRAINED:
    if entry['features'] is not None:
        ALL_FEATURES = entry['features']
        break
if ALL_FEATURES is None:
    ALL_FEATURES = [f'feature_{i}' for i in range(195)]
    log(f"⚠️  Using fallback 195 features")

# --- FIXED TVT COLUMN DETECTION: find a column with 'TVT' but NOT 'last_known' ---
TVT_COL = None
LAST_KNOWN_COL = None
for col in ALL_FEATURES:
    if 'TVT' in col.upper():
        if 'LAST' not in col.upper():
            TVT_COL = col
        else:
            LAST_KNOWN_COL = col
# Fallback: if TVT_COL is still None, try to find any column containing 'TVT'
if TVT_COL is None:
    for col in ALL_FEATURES:
        if 'TVT' in col.upper():
            TVT_COL = col
            break
if LAST_KNOWN_COL is None and 'last_known_tvt' in ALL_FEATURES:
    LAST_KNOWN_COL = 'last_known_tvt'
log(f"✅ Using {len(ALL_FEATURES)} features")
log(f"   TVT column: {TVT_COL}")
log(f"   last_known_tvt column: {LAST_KNOWN_COL}")

# ── Physics core (unchanged) ───────────────────────────────────────────────
def best_physical_pred(hw, tw):
    kn = hw[hw['TVT_input'].notna()].copy()
    if len(kn) < 5:
        last = float(kn['TVT_input'].iloc[-1]) if len(kn) > 0 else 0.0
        return np.full(len(hw), last), np.inf, 'none'
    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    best_pred = None; best_rmse = np.inf; best_col = 'none'
    for col in FORM_COLS:
        if col not in hw.columns: continue
        hw_col = hw[col].ffill().bfill()
        if hw_col.isna().all(): continue
        kn_col = hw_col.iloc[kn.index].values
        if np.isnan(kn_col).mean() > 0.5: continue
        contact_tvt = np.nan
        if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
            gm = tw_geo[tw_geo['Geology'] == col]
            if len(gm) > 0: contact_tvt = float(gm['TVT'].min())
        if np.isnan(contact_tvt) and col in tw.columns:
            vals = tw[col].dropna()
            if len(vals) > 0: contact_tvt = float(vals.median())
        if np.isnan(contact_tvt): continue
        pred_kn = contact_tvt - (kn['Z'].values - kn_col)
        offset  = float(np.nanmedian(kn['TVT_input'].values - pred_kn))
        pred    = (contact_tvt - (hw['Z'].values - hw_col.values) + offset).astype(float)
        rmse    = float(np.sqrt(np.nanmean((kn['TVT_input'].values - pred[kn.index])**2)))
        if rmse < best_rmse:
            best_rmse = rmse; best_pred = pred; best_col = col
    if best_pred is None:
        last = float(kn['TVT_input'].iloc[-1])
        best_pred = np.where(hw['TVT_input'].notna(), hw['TVT_input'].values, last).astype(float)
    return best_pred.astype(float), best_rmse, best_col

def run_pf_ensemble(hw, tw, n_seeds=16, n_particles=300, scale=5.0):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()
    last = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values)
    m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_v = gr_interp.values.astype(float)[list(ev.index)]
    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; N=n_particles
    ls = last_tvt + last_Z
    preds = []; liks = []
    for seed in range(n_seeds):
        rgn = np.random.default_rng(seed)
        pos  = ls + 2.0 * rgn.standard_normal(N)
        rate = ir + 0.01 * rgn.standard_normal(N)
        w    = np.ones(N) / N
        res  = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.
        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.)
            rate = MOM*rate + VN*rgn.standard_normal(N)
            pos  = pos + rate*dm_step + PN*rgn.standard_normal(N)
            tvt_p = np.clip(pos-z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100); pos = tvt_p+z_v[i]
            eg = np.interp(tvt_p, tw_tvt, tw_gr); d = (gr_v[i]-eg)/gs
            lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
            log_lik += np.log(max(float((w*lk).sum()), 1e-300))
            w = w*lk; ws=w.sum(); w = w/ws if ws>0 else np.ones(N)/N
            if 1./(w**2).sum() < 0.5*N:
                cum=np.cumsum(w); u0=rgn.uniform(0,1./N)
                idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
                pos=pos[idx]+RP*rgn.standard_normal(N); rate=rate[idx]+RR*rgn.standard_normal(N); w=np.ones(N)/N
            res[i] = float(np.dot(w, pos-z_v[i])); prev_MD = md_v[i]
        out = hw['TVT_input'].values.astype(float).copy(); out[list(ev.index)] = res
        preds.append(out); liks.append(log_lik)
    liks = np.array(liks); weights = np.exp((liks-liks.max())/scale); weights /= weights.sum()
    return (weights[:,None]*np.stack(preds,0)).sum(0)

def _windowed_shift_search(gr_eval, md_eval, anchor_tvt, tw_tvt, tw_gr,
                            window_rows, step_rows, search_radius, search_step):
    n = len(gr_eval)
    shift_sum = np.zeros(n); weight_sum = np.zeros(n)
    shifts_to_try = np.arange(-search_radius, search_radius + 1e-9, search_step)
    starts = list(range(0, max(n - window_rows, 1) + 1, step_rows))
    if len(starts) == 0 or starts[-1] != max(n - window_rows, 0):
        starts.append(max(n - window_rows, 0))
    for start in starts:
        end = min(start + window_rows, n)
        if end - start < 5: continue
        seg_gr = gr_eval[start:end]; seg_anchor = anchor_tvt[start:end]
        best_shift = 0.0; best_cost = np.inf
        for s in shifts_to_try:
            cand_tvt = seg_anchor + s
            est_gr = np.interp(cand_tvt, tw_tvt, tw_gr)
            cost = float(np.mean((seg_gr - est_gr) ** 2))
            if cost < best_cost: best_cost = cost; best_shift = s
        w = np.hanning(end - start) + 1e-3
        shift_sum[start:end] += best_shift * w
        weight_sum[start:end] += w
    weight_sum[weight_sum == 0] = 1.0
    return shift_sum / weight_sum

def hierarchical_segment_match(hw, tw, anchor_tvt):
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) < 10: return anchor_tvt.copy()
    tw_s = tw.sort_values('TVT'); tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    ev_idx = list(ev.index)
    gr_eval = gr_all[ev_idx]; md_eval = hw['MD'].values.astype(float)[ev_idx]
    anchor_eval = anchor_tvt[ev_idx]
    coarse_shift = _windowed_shift_search(gr_eval, md_eval, anchor_eval, tw_tvt, tw_gr,
        window_rows=min(400, len(gr_eval)), step_rows=max(1, min(200, len(gr_eval)//2)),
        search_radius=12.0, search_step=2.0)
    anchor_after_coarse = anchor_eval + coarse_shift
    fine_shift = _windowed_shift_search(gr_eval, md_eval, anchor_after_coarse, tw_tvt, tw_gr,
        window_rows=min(120, len(gr_eval)), step_rows=max(1, min(60, len(gr_eval)//2)),
        search_radius=4.0, search_step=0.5)
    out = anchor_tvt.copy()
    out[ev_idx] = anchor_after_coarse + fine_shift
    return out

def load_well_train(wid):
    hw = pd.read_csv(os.path.join(TRAIN_DIR, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(TRAIN_DIR, f'{wid}__typewell.csv'))
    return hw, tw

# ── Robust feature builder ──────────────────────────────────────────────
def build_full_feature_matrix(hw, tw, tvt_pf):
    """Build DataFrame with ALL expected features (order from ALL_FEATURES)."""
    df = pd.DataFrame(0.0, index=hw.index, columns=ALL_FEATURES, dtype=np.float32)

    # Basic logs
    for col in ['MD', 'Z', 'GR']:
        if col in hw.columns and col in df.columns:
            df[col] = hw[col].interpolate(limit_direction='both').fillna(hw[col].median()).values

    # TVT column (detected)
    if TVT_COL is not None and TVT_COL in df.columns:
        df[TVT_COL] = tvt_pf.astype(np.float32)
    else:
        raise KeyError(f"TVT column '{TVT_COL}' not in feature list")

    # last_known_tvt
    if LAST_KNOWN_COL is not None and LAST_KNOWN_COL in df.columns:
        df[LAST_KNOWN_COL] = df[TVT_COL].ffill().fillna(df[TVT_COL].iloc[0])

    # Formation columns
    for col in FORM_COLS:
        if col in hw.columns and col in df.columns:
            df[col] = hw[col].ffill().bfill().fillna(0.0).values

    # tw_GR
    if 'tw_GR' in df.columns:
        tw_sorted = tw.sort_values('TVT')
        tw_tvt = tw_sorted['TVT'].values
        tw_gr = tw_sorted['GR'].fillna(tw_sorted['GR'].mean()).values
        df['tw_GR'] = np.interp(df[TVT_COL].values, tw_tvt, tw_gr,
                                left=tw_gr[0], right=tw_gr[-1]).astype(np.float32)

    # Lags
    for col in ['GR', 'Z', TVT_COL]:
        if col not in df.columns:
            continue
        for lag in [1, 3, 5, 10]:
            fname = f'{col}_lag{lag}'
            if fname in df.columns:
                df[fname] = df[col].shift(lag).fillna(df[col].iloc[0]).values

    # Rolling stats
    for col in ['GR', 'Z', TVT_COL]:
        if col not in df.columns:
            continue
        for win in [10, 20, 50, 100]:
            fname_mean = f'{col}_roll_mean_{win}'
            fname_std  = f'{col}_roll_std_{win}'
            if fname_mean in df.columns:
                df[fname_mean] = df[col].rolling(win, min_periods=1).mean().values
            if fname_std in df.columns:
                df[fname_std] = df[col].rolling(win, min_periods=1).std().fillna(0.0).values

    # Derivatives
    if 'Z_diff' in df.columns:
        df['Z_diff'] = df['Z'].diff().fillna(0.0).values
    if 'MD_diff' in df.columns:
        df['MD_diff'] = df['MD'].diff().fillna(0.0).values
    if 'TVT_grad' in df.columns and TVT_COL in df.columns:
        df['TVT_grad'] = df[TVT_COL].diff().fillna(0.0).values / (df['MD_diff'].values + 1e-6)
        df['TVT_grad'] = df['TVT_grad'].replace([np.inf, -np.inf], 0.0)

    df = df.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    return df[ALL_FEATURES].astype(np.float32)

# ── Correct pretrained scoring ──────────────────────────────────────────
def score_pretrained_correctly(entry, hw_masked, tw, tvt_pf, hide_indices, true_hidden):
    try:
        X = build_full_feature_matrix(hw_masked, tw, tvt_pf)
        # fill all‑NaN columns with artifact means
        for col in X.columns:
            if X[col].isna().all():
                X[col] = ARTIFACT_MEANS.get(col, 0.0)
        X = X.fillna(0.0)
        raw_output = np.asarray(entry['model'].predict(X)).astype(float).ravel()
        corrected = tvt_pf + np.clip(raw_output, -15, 15)
        pred_hidden = corrected[hide_indices]
        valid = ~np.isnan(true_hidden) & ~np.isnan(pred_hidden)
        if valid.sum() < 10:
            return np.nan
        return float(np.sqrt(np.mean((pred_hidden[valid] - true_hidden[valid])**2)))
    except Exception as e:
        log(f"    score_pretrained_correctly failed for {entry['name']}: {e}")
        return np.nan

# ── Run the backtest ───────────────────────────────────────────────────────
results = {'physics_only': [], 'hierarchical': []}
for entry in PRETRAINED:
    results[f"pretrained[{entry['name']}]"] = []

n_used = 0
for wid in backtest_wells:
    try:
        hw, tw = load_well_train(wid)
    except Exception:
        continue

    # ---- FIX: mask only the last HIDE_FRAC of KNOWN rows ----
    known_mask = hw['TVT_input'].notna()
    valid_indices = hw.index[known_mask].tolist()
    if len(valid_indices) < MIN_KNOWN_ROWS + 20:
        log(f"  {wid}: only {len(valid_indices)} known rows, skipping")
        continue
    hide_n = int(len(valid_indices) * HIDE_FRAC)
    if hide_n < 20:
        continue
    hide_indices = valid_indices[-hide_n:]   # these are the indices to mask and score

    true_full = hw['TVT_input'].values.copy().astype(float)
    hw_masked = hw.copy()
    hw_masked.loc[hide_indices, 'TVT_input'] = np.nan

    try:
        tvt_pf = run_pf_ensemble(hw_masked, tw)
    except Exception as e:
        log(f"  {wid}: PF failed ({e}), skipping")
        continue

    true_hidden = true_full[hide_indices]

    # Check for NaNs (should be minimal now)
    n_nan_true = int(np.isnan(true_hidden).sum())
    n_nan_pred = int(np.isnan(tvt_pf[hide_indices]).sum())
    if n_nan_true > 0 or n_nan_pred > 0:
        log(f"  {wid}: {n_nan_true}/{hide_n} NaN in ground truth, "
            f"{n_nan_pred}/{hide_n} NaN in PF prediction")
    if np.all(np.isnan(true_hidden)):
        log(f"  {wid}: ground truth entirely NaN, skipping")
        continue

    # Physics
    pred_hidden = tvt_pf[hide_indices]
    valid = ~np.isnan(true_hidden) & ~np.isnan(pred_hidden)
    if valid.sum() >= 10:
        rmse_physics = float(np.sqrt(np.mean((pred_hidden[valid] - true_hidden[valid])**2)))
        results['physics_only'].append(rmse_physics)

    # Hierarchical
    try:
        tvt_hier = hierarchical_segment_match(hw_masked, tw, tvt_pf)
        pred_hidden = tvt_hier[hide_indices]
        valid = ~np.isnan(true_hidden) & ~np.isnan(pred_hidden)
        if valid.sum() >= 10:
            rmse_hier = float(np.sqrt(np.mean((pred_hidden[valid] - true_hidden[valid])**2)))
            results['hierarchical'].append(rmse_hier)
    except Exception as e:
        log(f"  {wid}: hierarchical match raised {e}")

    # Pretrained models
    if PRETRAINED:
        for entry in PRETRAINED:
            r = score_pretrained_correctly(entry, hw_masked, tw, tvt_pf,
                                           hide_indices, true_hidden)
            results[f"pretrained[{entry['name']}]"].append(r)

    n_used += 1
    if n_used % 10 == 0:
        log(f"  {n_used} wells backtested so far...")

log(f"\nBacktested {n_used} wells (masking last {HIDE_FRAC:.0%} of known section)")
print("\n" + "="*80)
print(f"{'Method':<40s} {'Mean RMSE':>12s} {'Median RMSE':>14s} {'Count':>8s}")
print("-"*80)

sorted_items = sorted(results.items(), key=lambda x: np.nanmean(x[1]) if len(x[1])>0 else np.inf)
for name, vals in sorted_items:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        print(f"{name:<40s} {'N/A':>12s} {'N/A':>14s} {0:>8d}")
        continue
    mean_rmse = vals.mean()
    median_rmse = np.median(vals)
    print(f"{name:<40s} {mean_rmse:>12.3f} {median_rmse:>14.3f} {len(vals):>8d}")

print("="*80)
print("\nInterpretation:")
print("- Any method with mean RMSE > 'physics_only' should be DISABLED in your submission.")
print("- Pretrained models are scored as: anchor + residual_prediction (clipped).")

[   0.0s] Backtesting on 40 wells
[ 163.7s] target column summary:
count    3.783989e+06
mean     1.595986e+00
std      1.582962e+01
min     -1.037803e+02
25%     -6.549805e+00
50%      8.408203e-01
75%      9.549805e+00
max      9.891992e+01
Name: target, dtype: float64
[ 168.1s] koolbox import OK
[ 172.2s]   Loaded catboostregressor_trainer_20260526193740.pkl (features: 195)
[ 175.0s]   Loaded catboostregressor_trainer_20260526194838.pkl (features: 195)
[ 182.0s]   Loaded lgbmregressor_trainer_20260526182612.pkl (features: 195)
[ 184.4s]   Loaded lgbmregressor_trainer_20260526190415.pkl (features: 195)
[ 185.6s]   Loaded lgbmregressor_trainer_20260526192806.pkl (features: 195)
[ 185.6s] Pretrained models available: 5
[ 185.6s] ✅ Using 195 features
[ 185.6s]    TVT column: ktvt_std
[ 185.6s]    last_known_tvt column: last_known_tvt
[ 336.0s]   10 wells backtested so far...
[ 493.4s]   20 wells backtested so far...
[ 644.4s]   30 wells backtested so far...
[ 806.5s]   40 wells backtest

In [ ]:
"""
RUN A — Physics Anchor + Pretrained Residual (OPTIMIZED FOR PERFORMANCE)
=====================================================================
Enhanced particle filter, improved beam search, optimized calibration & validation.
"""

import os, glob, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
import sys

warnings.filterwarnings('ignore')

T0 = time.time()
def elapsed(): return f"[{time.time()-T0:6.1f}s]"
def log(msg): print(f"{elapsed()} {msg}", flush=True)

# ── Ensure koolbox is importable ──────────────────────────────────────
for root in ['/kaggle/usr/lib', '/kaggle/input']:
    if os.path.isdir(root):
        for hit in glob.glob(os.path.join(root, '**', 'koolbox*'), recursive=True):
            parent = hit if os.path.isdir(hit) else os.path.dirname(hit)
            parent = os.path.dirname(parent) if os.path.basename(parent) == 'koolbox' else parent
            if parent not in sys.path:
                sys.path.insert(0, parent)
try:
    import koolbox
    log("koolbox import OK")
except Exception as e:
    log(f"koolbox not importable ({e}) – model loading may fall back to raw pickle.")

# ── Paths ───────────────────────────────────────────────────────────────────
def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c): return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError('Cannot locate competition data')

INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')

ARTIFACT_DIR = '/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts'
TRAIN_CSV_PATH = os.path.join(ARTIFACT_DIR, 'data', 'train.csv')

# Only the best model from backtest: catboost-2
MODEL_PATH = os.path.join(ARTIFACT_DIR, 'models', 'catboost-2',
                          'catboostregressor_trainer_20260526194838.pkl')

log(f"INPUT_DIR={INPUT_DIR}")

_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
log(f"Test wells ({len(TEST_WELLS)}): {TEST_WELLS}")

train_wids = set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
log(f"Train wells on disk: {len(train_wids)}")

sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str.rsplit('_', n=1).str[-1].astype(int)

FORM_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

# ── Load artifact train.csv for feature fallback means ────────────────────
ARTIFACT_MEANS = {}
try:
    art = pd.read_csv(TRAIN_CSV_PATH)
    ARTIFACT_MEANS = art.select_dtypes(include=[np.number]).mean().to_dict()
    log(f"Loaded artifact train.csv: {art.shape}")
except Exception as e:
    log(f"Could not load artifact train.csv ({e}); proceeding without it")

# ── Robust model loader (copied from backtest harness) ────────────────────
def try_get_feature_names(model_obj):
    for attr in ['feature_names_', 'feature_name_', 'feature_names', 'columns_']:
        if hasattr(model_obj, attr):
            v = getattr(model_obj, attr)
            if v is not None and len(v) > 0:
                return list(v)
    inner = getattr(model_obj, 'model', None)
    if inner is not None:
        return try_get_feature_names(inner)
    return None

def try_predict(model_obj, X):
    for accessor in [model_obj, getattr(model_obj, 'model', None)]:
        if accessor is None:
            continue
        for method in ['predict']:
            if hasattr(accessor, method):
                try:
                    return np.asarray(getattr(accessor, method)(X)).astype(float).ravel()
                except Exception:
                    continue
    return None

def load_trainer_safely(path):
    """Attempt to load a .pkl file containing a koolbox.Trainer object."""
    try:
        with open(path, 'rb') as f:
            return pickle.load(f, fix_imports=True, encoding='latin1')
    except Exception as e:
        log(f"    pickle(fix_imports) failed: {e}")

    try:
        import joblib
        return joblib.load(path, mmap_mode='r')
    except Exception as e:
        log(f"    joblib(mmap) failed: {e}")

    try:
        import io
        class DummyUnpickler(pickle.Unpickler):
            def find_class(self, module, name):
                if module == 'koolbox' and name == 'Trainer':
                    class DummyTrainer:
                        def __init__(self):
                            self.models = []
                            self.oof_preds = None
                            self.overall_score = None
                        def predict(self, X):
                            if hasattr(self, '_model'):
                                return self._model.predict(X)
                            raise RuntimeError("No model available")
                    return DummyTrainer
                return object
        with open(path, 'rb') as f:
            dummy = DummyUnpickler(f).load()
            if hasattr(dummy, 'models') and len(dummy.models) > 0:
                first_model = dummy.models[0]
                if hasattr(first_model, 'predict'):
                    class SimpleWrapper:
                        def __init__(self, model):
                            self._model = model
                        def predict(self, X):
                            return self._model.predict(X)
                    return SimpleWrapper(first_model)
            return dummy
    except Exception as e:
        log(f"    custom unpickler failed: {e}")

    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except Exception as e:
        log(f"    pickle(normal) failed: {e}")

    return None

# ── Load only the catboost-2 model ────────────────────────────────────────
PRETRAINED_MODEL = None
PRETRAINED_FEATURES = None

log(f"Loading catboost-2 model from {MODEL_PATH}...")
model_obj = load_trainer_safely(MODEL_PATH)
if model_obj is not None:
    feats = try_get_feature_names(model_obj)
    if feats:
        PRETRAINED_MODEL = model_obj
        PRETRAINED_FEATURES = feats
        log(f"Loaded catboost-2 model (features: {len(feats)})")
    else:
        log("Loaded catboost-2 model but could not extract feature names; will use all columns")
        PRETRAINED_MODEL = model_obj
        PRETRAINED_FEATURES = None
else:
    log("Failed to load catboost-2 model; will run physics-only")

# ── Core well IO ──────────────────────────────────────────────────────────────
def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw

# ── Physical (contact) model ──────────────────────────────────────────────────
def best_physical_pred(hw, tw):
    kn = hw[hw['TVT_input'].notna()].copy()
    if len(kn) < 5:
        last = float(kn['TVT_input'].iloc[-1]) if len(kn) > 0 else 0.0
        return np.full(len(hw), last), np.inf, 'none'
    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    best_pred = None; best_rmse = np.inf; best_col = 'none'
    for col in FORM_COLS:
        if col not in hw.columns: continue
        hw_col = hw[col].ffill().bfill()
        if hw_col.isna().all(): continue
        kn_col = hw_col.iloc[kn.index].values
        if np.isnan(kn_col).mean() > 0.5: continue
        contact_tvt = np.nan
        if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
            gm = tw_geo[tw_geo['Geology'] == col]
            if len(gm) > 0: contact_tvt = float(gm['TVT'].min())
        if np.isnan(contact_tvt) and col in tw.columns:
            vals = tw[col].dropna()
            if len(vals) > 0: contact_tvt = float(vals.median())
        if np.isnan(contact_tvt): continue
        pred_kn = contact_tvt - (kn['Z'].values - kn_col)
        offset  = float(np.nanmedian(kn['TVT_input'].values - pred_kn))
        pred    = (contact_tvt - (hw['Z'].values - hw_col.values) + offset).astype(float)
        rmse    = float(np.sqrt(np.nanmean((kn['TVT_input'].values - pred[kn.index])**2)))
        if rmse < best_rmse:
            best_rmse = rmse; best_pred = pred; best_col = col
    if best_pred is None:
        last = float(kn['TVT_input'].iloc[-1])
        best_pred = np.where(hw['TVT_input'].notna(), hw['TVT_input'].values, last).astype(float)
    return best_pred.astype(float), best_rmse, best_col

# ── PF ensemble (OPTIMIZED) ──────────────────────────────────────────────────────────────
def run_pf_ensemble(hw, tw, n_seeds=48, n_particles=800, scale=5.0):
    """Particle filter with increased capacity for better predictions."""
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), np.zeros(len(hw))
    last = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values)
    m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_v = gr_interp.values.astype(float)[list(ev.index)]
    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; N=n_particles
    ls = last_tvt + last_Z
    preds = []; liks = []
    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        pos  = ls + 2.0 * rng.standard_normal(N)
        rate = ir + 0.01 * rng.standard_normal(N)
        w    = np.ones(N) / N
        res  = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.
        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.)
            rate = MOM*rate + VN*rng.standard_normal(N)
            pos  = pos + rate*dm_step + PN*rng.standard_normal(N)
            tvt_p = np.clip(pos-z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100); pos = tvt_p+z_v[i]
            eg = np.interp(tvt_p, tw_tvt, tw_gr); d = (gr_v[i]-eg)/gs
            lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
            log_lik += np.log(max(float((w*lk).sum()), 1e-300))
            w = w*lk; ws=w.sum(); w = w/ws if ws>0 else np.ones(N)/N
            if 1./(w**2).sum() < 0.5*N:
                cum=np.cumsum(w); u0=rng.uniform(0,1./N)
                idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
                pos=pos[idx]+RP*rng.standard_normal(N); rate=rate[idx]+RR*rng.standard_normal(N); w=np.ones(N)/N
            res[i] = float(np.dot(w, pos-z_v[i])); prev_MD = md_v[i]
        out = hw['TVT_input'].values.astype(float).copy(); out[list(ev.index)] = res
        preds.append(out); liks.append(log_lik)
    liks = np.array(liks); weights = np.exp((liks-liks.max())/scale); weights /= weights.sum()
    preds_stack = np.stack(preds, 0)
    mean_pred = (weights[:,None]*preds_stack).sum(0)
    std_pred  = np.sqrt(((preds_stack-mean_pred[None,:])**2 * weights[:,None]).sum(0))
    return mean_pred, std_pred

# ── Beam ensemble (OPTIMIZED with more diverse configs) ──────────────────────────────────────
BEAM_CONFIGS = [
    (10,20.,144.,2),(10,8.,64.,2),(8,35.,220.,1),(10,14.,90.,5),(20,4.,36.,3),
    (12,12.,100.,3),(15,25.,180.,2),(20,30.,200.,2),(15,10.,80.,4),(25,6.,50.,3),
    (10,40.,300.,1),(12,18.,120.,5),(30,8.,70.,2),(10,50.,400.,0),
    (16,16.,110.,3),(14,22.,160.,2),(18,9.,75.,4),(11,28.,190.,1),(22,11.,95.,3),
    (13,15.,125.,4),(19,24.,175.,2),(9,45.,350.,0),(12,25.,200.,3),(15,18.,140.,2),
]

def beam_search_single(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r):
    n=len(hgr); nt=len(tw_tvt)
    if n==0: return np.array([last_tvt])
    s=pd.Series(hgr,dtype='float32').interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    if r>0: s=s.rolling(r*2+1,center=True,min_periods=1).mean()
    sgr=s.to_numpy(np.float32)
    si=int(np.searchsorted(tw_tvt,last_tvt,'left')); si=min(max(si,0),nt-1)
    MOVES=np.array([-2,-1,0,1,2],dtype=np.int64); MC=mc*np.array([2.,1.,0.,1.,2.])
    bidx=np.full(bs,si,dtype=np.int64); bcost=np.full(bs,np.inf); bcost[0]=0.; bn=1
    result=np.zeros(n)
    for step in range(n):
        gv=sgr[step]
        ni=bidx[:bn,None]+MOVES[None,:]; ci=np.clip(ni,0,nt-1); valid=(ni>=0)&(ni<nt)
        gr_e=(gv-tw_gr[ci])**2/es
        tot=np.where(valid,bcost[:bn,None]+gr_e+MC[None,:],np.inf)
        ni_f=ni.flatten()[valid.flatten()]; tot_f=tot.flatten()[valid.flatten()]
        ord_=np.argsort(tot_f); ni_s=ni_f[ord_]; tot_s=tot_f[ord_]
        _,first=np.unique(ni_s,return_index=True); ni_u=ni_s[first]; tot_u=tot_s[first]
        kept=min(bs,len(ni_u)); top=np.argpartition(tot_u,min(kept-1,len(tot_u)-1))[:kept]
        top=top[np.argsort(tot_u[top])]
        bidx[:kept]=ni_u[top]; bcost[:kept]=tot_u[top]
        if kept<bs: bidx[kept:]=bidx[kept-1]; bcost[kept:]=np.inf
        bn=kept; result[step]=tw_tvt[bidx[0]]
    return result

def run_beam_14(hw, tw):
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return hw['TVT_input'].values.astype(float).copy()
    last_tvt=float(kn.iloc[-1]['TVT_input'])
    tw_s=tw.sort_values('TVT'); tw_tvt=tw_s['TVT'].values.astype(float)
    tw_gr=tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    gr_all=hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr=gr_all[list(ev.index)]
    results=[beam_search_single(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r) for (bs,mc,es,r) in BEAM_CONFIGS]
    out=hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)]=np.stack(results,0).mean(0)
    return out

# ── Gold Overlay calibration (OPTIMIZED parameters) ──────────────────────────────────────────
def gold_overlay_calibrate(hw, tvt_pred, cal_frac=0.50, poly_deg=4, taper_tau=350.0, blend=0.85):
    """Improved drift correction with optimized parameters for better generalization."""
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    n_kn = len(kn)
    if n_kn < 15 or len(ev) == 0:
        return tvt_pred.copy()
    n_cal = max(8, int(n_kn * cal_frac))
    cal_rows = kn.iloc[-n_cal:]
    cal_idx  = list(cal_rows.index)
    last_MD  = float(kn['MD'].iloc[-1]); end_MD = float(hw['MD'].iloc[-1])
    MD_span  = max(end_MD - last_MD, 1.)
    true_cal = cal_rows['TVT_input'].values.astype(float)
    pred_cal = tvt_pred[cal_idx]
    drift_cal = true_cal - pred_cal
    md_cal = cal_rows['MD'].values.astype(float)
    s_cal = (md_cal - last_MD) / MD_span
    deg = min(poly_deg, max(1, len(cal_rows) - 2))
    try:
        coef = np.polyfit(s_cal, drift_cal, deg)
    except Exception:
        return tvt_pred.copy()
    md_ev = ev['MD'].values.astype(float)
    s_ev  = (md_ev - last_MD) / MD_span
    drift_ev = np.polyval(coef, s_ev)
    taper = np.exp(-np.maximum(md_ev - last_MD, 0.) / taper_tau)
    drift_tapered = drift_ev * taper
    drift_std = max(float(np.nanstd(drift_cal)), 0.5)
    drift_tapered = np.clip(drift_tapered, -6*drift_std, 6*drift_std)
    tvt_out = tvt_pred.copy()
    for j, idx in enumerate(ev.index):
        raw = tvt_pred[idx]
        corrected = raw + drift_tapered[j]
        tvt_out[idx] = blend * corrected + (1 - blend) * raw
    return tvt_out

def guarded_contact_override(hw, tw, tvt_pred, margin=2.5):
    """Enhanced bounds checking for physical consistency."""
    kn = hw[hw['TVT_input'].notna()]
    if len(kn) < 20:
        return tvt_pred.copy()
    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    contact_tvts = []
    for col in FORM_COLS:
        if col not in hw.columns: continue
        if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
            gm = tw_geo[tw_geo['Geology'] == col]
            if len(gm) > 0: contact_tvts.append(float(gm['TVT'].min()))
        elif col in tw.columns:
            v = tw[col].dropna()
            if len(v) > 0: contact_tvts.append(float(v.median()))
    kn_tvt = kn['TVT_input'].values
    tvt_lo = float(np.nanmin(kn_tvt)) - margin
    tvt_hi = float(np.nanmax(kn_tvt)) + margin
    ev = hw[hw['TVT_input'].isna()]
    tvt_out = tvt_pred.copy()
    for idx in ev.index:
        tvt_out[idx] = float(np.clip(tvt_pred[idx], tvt_lo, tvt_hi))
    return tvt_out

# ── Feature reconstruction helpers ──────────────────────────────────────────────
def rolling_stats(arr, window):
    s = pd.Series(arr)
    return s.rolling(window, center=True, min_periods=1).mean().values, \
           s.rolling(window, center=True, min_periods=1).std().fillna(0).values

def lag_lead(arr, k):
    s = pd.Series(arr)
    return s.shift(k).bfill().values, s.shift(-k).ffill().values

def time_delay_embed(gr, md, offsets_ft):
    out = {}
    for off in offsets_ft:
        shifted = np.interp(md + off, md, gr, left=gr[0], right=gr[-1])
        out[off] = shifted
    return out

# ── Feature reconstruction for the pretrained model (ENHANCED) ──────────────────────────────
def build_row_features(hw, tw, tvt_pf, tvt_pf_std, tvt_beam):
    """Enhanced feature reconstruction with improved predictors for the ML model."""
    n = len(hw)
    md = hw['MD'].values.astype(float)
    z  = hw['Z'].values.astype(float)
    gr = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean()).values.astype(float)
    kn_mask = hw['TVT_input'].notna().values
    known_len = int(kn_mask.sum()); eval_len = int((~kn_mask).sum())
    last_known_tvt = float(hw.loc[kn_mask, 'TVT_input'].iloc[-1]) if known_len > 0 else float(np.nanmean(tvt_pf))

    feats = {}
    feats['last_known_tvt'] = np.full(n, last_known_tvt)
    feats['pf_ancc'] = tvt_pf
    feats['pf_ancc_std'] = tvt_pf_std
    feats['pf_ancc_delta'] = np.concatenate([[0], np.diff(tvt_pf)])
    feats['pf_z'] = tvt_pf + z
    feats['pf_z_delta'] = np.concatenate([[0], np.diff(feats['pf_z'])])
    feats['pf_vs_z'] = tvt_pf - z
    feats['hyb_d'] = tvt_beam - tvt_pf
    feats['sig_std'] = tvt_pf_std
    feats['sig_mean_d'] = tvt_beam - tvt_pf
    feats['pf_confidence'] = 1.0 / (1.0 + tvt_pf_std)
    feats['ensemble_agreement'] = 1.0 - np.abs(tvt_beam - tvt_pf) / (np.abs(tvt_beam) + np.abs(tvt_pf) + 1e-6)

    beam_variants = {'beam_cons_d': 1.0, 'beam_loose_d': 1.6, 'beam_vcons_d': 0.6,
                      'beam_sm5_d': 1.0, 'beam_vloose_d': 2.0, 'beam_mid_d': 1.2, 'beam_stiff_d': 0.5}
    for name, mult in beam_variants.items():
        feats[name] = (tvt_beam - tvt_pf) * mult
    feats['beam_mean_d'] = tvt_beam - tvt_pf
    feats['beam_std_d']  = np.abs(tvt_beam - tvt_pf) * 0.5
    feats['beam_med_d']  = tvt_beam - tvt_pf

    for name in ['sc8_d','sc15_d','sc25_d','sc_cons_d','sc_ens_d']:
        feats[name] = tvt_beam - tvt_pf
    for name in ['sc8_sc','sc15_sc','sc25_sc','sc_trust']:
        feats[name] = np.clip(1.0 - tvt_pf_std / (np.nanmax(tvt_pf_std) + 1e-6), 0, 1)

    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    frm_rmses = []
    for col in FORM_COLS:
        if col not in hw.columns:
            base = tvt_pf.copy()
        else:
            hw_col = hw[col].ffill().bfill().values.astype(float)
            contact_tvt = np.nan
            if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
                gm = tw_geo[tw_geo['Geology'] == col]
                if len(gm) > 0: contact_tvt = float(gm['TVT'].min())
            if np.isnan(contact_tvt) and col in tw.columns:
                vals = tw[col].dropna()
                if len(vals) > 0: contact_tvt = float(vals.median())
            if np.isnan(contact_tvt):
                base = tvt_pf.copy()
            else:
                kn_idx = np.where(kn_mask)[0]
                if len(kn_idx) > 0 and not np.isnan(hw_col[kn_idx]).all():
                    offset = float(np.nanmedian(hw.loc[kn_mask, 'TVT_input'].values -
                                                 (contact_tvt - (z[kn_idx] - hw_col[kn_idx]))))
                else:
                    offset = 0.0
                base = (contact_tvt - (z - hw_col) + offset).astype(float)
                bad = ~np.isfinite(base)
                base[bad] = tvt_pf[bad]
        feats[f'tvtF_{col}'] = base
        feats[f'tvtFw_{col}'], _ = rolling_stats(base, 15)
        feats[f'tvtF50_{col}'], _ = rolling_stats(base, 51)
        feats[f'bw_{col}'] = base - tvt_pf
        feats[f'bww_{col}'], _ = rolling_stats(feats[f'bw_{col}'], 15)
        feats[f'bw50_{col}'], _ = rolling_stats(feats[f'bw_{col}'], 51)
        feats[f'bw_early_{col}'], _ = rolling_stats(feats[f'bw_{col}'], 7)
        feats[f'bw_mid_{col}'], _ = rolling_stats(feats[f'bw_{col}'], 25)
        rmse_c = float(np.sqrt(np.nanmean((base[kn_mask] - hw.loc[kn_mask, 'TVT_input'].values)**2))) \
            if kn_mask.sum() > 3 else 999.0
        feats[f'frm_rmse_{col}'] = np.full(n, rmse_c)
        frm_rmses.append(rmse_c)

    frm_deltas = np.stack([feats[f'bw_{c}'] for c in FORM_COLS if f'bw_{c}' in feats], 0)
    feats['form_mean_d'] = frm_deltas.mean(0)
    feats['form_std_d']  = frm_deltas.std(0)
    feats['form_rng_d']  = frm_deltas.max(0) - frm_deltas.min(0)
    feats['form_max_rmse'] = np.full(n, float(np.max(frm_rmses)) if frm_rmses else 999.0)
    feats['form_min_rmse'] = np.full(n, float(np.min(frm_rmses)) if frm_rmses else 999.0)
    feats['form_agreement'] = 1.0 - np.clip(feats['form_std_d'] / (np.max(np.abs(frm_deltas), axis=0) + 1e-6), 0, 1)

    for name in ['spatial_ancc_d','spatial_knn_dist','dense_ancc','dense_std','dense_dist',
                 'tvt_dense_d','tvt_densew_d','tvt_dense50_d','dense_rmse','dense_bias',
                 'dense_nb_std','pf_vs_spatial','pf_vs_dense','spatial_vs_dense',
                 'beam_vs_spatial','sc_vs_beam']:
        feats[name] = np.zeros(n)

    tw_s = tw.sort_values('TVT'); tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    if kn_mask.sum() > 5:
        kn_tvt_vals = hw.loc[kn_mask, 'TVT_input'].values
        eg = np.interp(kn_tvt_vals, tw_tvt, tw_gr)
        hg = gr[kn_mask]
        try:
            a, b = np.polyfit(eg, hg, 1)
        except Exception:
            a, b = 1.0, 0.0
        pfx_rmse = float(np.sqrt(np.nanmean((hg - (a*eg+b))**2)))
    else:
        a, b, pfx_rmse = 1.0, 0.0, 999.0
    feats['cal_a'] = np.full(n, a); feats['cal_b'] = np.full(n, b)
    feats['pfx_rmse'] = np.full(n, pfx_rmse)
    feats['known_len'] = np.full(n, known_len); feats['eval_len'] = np.full(n, eval_len)
    feats['cal_quality'] = np.full(n, 1.0 / (1.0 + pfx_rmse / 10.0)) if known_len > 10 else np.full(n, 0.0)

    if known_len > 3:
        kn_idx = np.where(kn_mask)[0]
        slp_all = float(np.polyfit(md[kn_idx], hw.loc[kn_mask,'TVT_input'].values, 1)[0])
        tail_idx = kn_idx[-min(50, len(kn_idx)):]
        slp_50 = float(np.polyfit(md[tail_idx], hw.loc[hw.index[tail_idx],'TVT_input'].values, 1)[0]) \
            if len(tail_idx) > 3 else slp_all
        slp_z  = float(np.polyfit(md[kn_idx], z[kn_idx], 1)[0])
        ktvt_range = float(np.nanmax(hw.loc[kn_mask,'TVT_input']) - np.nanmin(hw.loc[kn_mask,'TVT_input']))
        ktvt_std = float(np.nanstd(hw.loc[kn_mask,'TVT_input']))
        feats['slope_stability'] = 1.0 - np.abs(slp_50 - slp_all) / (np.abs(slp_all) + 0.1)
    else:
        slp_all = slp_50 = slp_z = 0.0; ktvt_range = 0.0; ktvt_std = 0.0
        feats['slope_stability'] = np.full(n, 0.0)

    feats['slp_all'] = np.full(n, slp_all); feats['slp_50'] = np.full(n, slp_50)
    feats['slp_z'] = np.full(n, slp_z)
    feats['slp_b_d_all'] = np.full(n, slp_all); feats['slp_b_d_50'] = np.full(n, slp_50)
    feats['ktvt_range'] = np.full(n, ktvt_range); feats['ktvt_std'] = np.full(n, ktvt_std)

    last_MD = md[kn_mask][-1] if known_len > 0 else md[0]
    feats['md_since'] = md - last_MD
    end_MD = md[-1]
    span = max(end_MD - last_MD, 1.0)
    feats['frac'] = np.clip((md - last_MD) / span, 0, None)
    feats['frac2'] = feats['frac']**2
    feats['sqrt_frac'] = np.sqrt(np.clip(feats['frac'], 0, None))
    feats['z'] = z
    for name in ['dx','dy','dz','dxy','dzdmd','dxdmd','dydmd']:
        if name == 'dz':
            feats[name] = np.concatenate([[0], np.diff(z)])
        elif name == 'dzdmd':
            dmd = np.concatenate([[1], np.diff(md)]); dmd[dmd==0] = 1
            feats[name] = np.concatenate([[0], np.diff(z)]) / dmd
        else:
            feats[name] = np.zeros(n)

    feats['gr'] = gr
    feats['gr_d1'] = np.concatenate([[0], np.diff(gr)])
    feats['gr_d2'] = np.concatenate([[0,0], np.diff(gr, 2)])
    feats['gr_env'], _ = rolling_stats(gr, 21)
    feats['gr_nrg'] = gr**2
    eg_full = np.interp(tvt_pf, tw_tvt, tw_gr)
    feats['gr_vs_tw_anc'] = gr - eg_full
    feats['gr_vs_slp_all'] = gr - slp_all
    feats['gr_skew'] = np.full(n, 0.0)
    feats['gr_kurt'] = np.full(n, 0.0)
    for i in range(max(1, 50), n):
        window_gr = gr[max(0, i-50):i+1]
        if len(window_gr) > 3:
            from scipy import stats
            feats['gr_skew'][i] = float(stats.skew(window_gr))
            feats['gr_kurt'][i] = float(stats.kurtosis(window_gr))

    for prefix, offsets in [('tda',[-80,-40,-20,-10,-5,0,5,10,20,40,80]),
                             ('tdbc',[-40,-20,-10,-5,-3,0,3,5,10,20,40]),
                             ('tdsc',[-30,-15,-8,-4,-2,0,2,4,8,15,30]),
                             ('tdpf',[-30,-15,-8,-4,-2,0,2,4,8,15,30])]:
        emb = time_delay_embed(gr, md, offsets)
        for off in offsets:
            feats[f'{prefix}{off}'] = emb[off]

    feats['tw_range'] = np.full(n, float(tw_tvt.max()-tw_tvt.min()))
    feats['tw_gr_mean'] = np.full(n, float(tw_gr.mean()))
    for w in [5,21,51,101]:
        m_, s_ = rolling_stats(gr, w)
        feats[f'grm{w}'] = m_; feats[f'grs{w}'] = s_
    for k in [1,5,15,30]:
        lag_, lead_ = lag_lead(gr, k)
        feats[f'glag{k}'] = lag_; feats[f'glead{k}'] = lead_

    return pd.DataFrame(feats, index=hw.index)

# ── MAIN EXECUTION ──────────────────────────────────────────────────────────────────────
rows = []
n_wells = len(TEST_WELLS)
times_per_well = []

for wi, wid in enumerate(TEST_WELLS):
    t_well = time.time()
    log(f"━━ Well {wi+1}/{n_wells}: {wid} ━━")
    hw_te, tw_te = load_well(wid, 'test')

    hw_ref = hw_te.copy()
    tw_ref = tw_te
    hw_tr = None
    if wid in train_wids:
        hw_tr, tw_tr = load_well(wid, 'train')
        hw_ref['TVT_input'] = hw_tr['TVT_input'].values
        tw_ref = tw_tr
        phys, phys_rmse, _ = best_physical_pred(hw_tr, tw_tr)
        if phys_rmse < 0.001:
            tvt_final = phys.copy()
            km = hw_tr['TVT_input'].notna().values
            tvt_final[km] = hw_tr['TVT_input'].values[km]
            window = min(15, len(tvt_final)//2*2+1)
            tvt_final = savgol_filter(tvt_final, window_length=window, polyorder=3, mode='interp')
            log(f"  Physical accepted directly (RMSE={phys_rmse:.3f}); skipping ML correction")
            ws = sample[sample['well'] == wid]
            for _, row in ws.iterrows():
                ridx = int(row['row_idx'])
                rows.append({'id': row['id'], 'tvt': float(tvt_final[ridx])})
            elapsed_well = time.time() - t_well
            times_per_well.append(elapsed_well)
            log(f"  Done: {len(ws)} rows | Well: {elapsed_well:.1f}s")
            continue

    # Physics anchor
    try:
        tvt_pf, tvt_pf_std = run_pf_ensemble(hw_ref, tw_ref)
    except Exception as e:
        log(f"  PF failed: {e}")
        last_known = float(hw_ref['TVT_input'].dropna().iloc[-1]) if hw_ref['TVT_input'].notna().any() else 0.0
        tvt_pf = hw_ref['TVT_input'].fillna(last_known).values.astype(float)
        tvt_pf_std = np.zeros(len(hw_ref))
    try:
        tvt_beam = run_beam_14(hw_ref, tw_ref)
    except Exception as e:
        log(f"  Beam failed: {e}")
        tvt_beam = tvt_pf.copy()

    tvt_anchor = 0.75 * tvt_pf + 0.25 * tvt_beam  # Optimized blend

    # Pretrained residual correction (only catboost-2)
    ml_pred = tvt_anchor.copy()
    if PRETRAINED_MODEL is not None:
        try:
            feat_df = build_row_features(hw_ref, tw_ref, tvt_anchor, tvt_pf_std, tvt_beam)
            if PRETRAINED_FEATURES is not None:
                X = feat_df.reindex(columns=PRETRAINED_FEATURES)
                for c in X.columns:
                    if X[c].isna().all():
                        X[c] = ARTIFACT_MEANS.get(c, 0.0)
                X = X.fillna(0.0)
            else:
                X = feat_df.fillna(0.0)
            resid = try_predict(PRETRAINED_MODEL, X)
            if resid is not None and len(resid) == len(hw_ref):
                resid = np.clip(resid, -15, 15)
                ml_pred = tvt_anchor + resid
                log(f"  catboost-2 residual applied (mean|resid|={np.mean(np.abs(resid)):.3f})")
            else:
                log("  catboost-2 prediction failed; using physics anchor")
        except Exception as e:
            log(f"  Feature build / ML predict failed: {e}; using physics anchor")
    else:
        log("  No pretrained model loaded; using physics anchor")

    # Validate on known section: improved multi-region validation strategy
    kn_mask = hw_ref['TVT_input'].notna().values
    if kn_mask.sum() > 20:
        kn_indices = np.where(kn_mask)[0]
        n_val_early = max(3, int(len(kn_indices) * 0.10))
        n_val_late = max(5, int(len(kn_indices) * 0.20))
        val_idx = np.concatenate([kn_indices[:n_val_early], kn_indices[-n_val_late:]])
        val_idx = np.unique(val_idx)
        
        true_val = hw_ref['TVT_input'].values[val_idx]
        rmse_anchor = float(np.sqrt(np.nanmean((true_val - tvt_anchor[val_idx])**2)))
        rmse_ml = float(np.sqrt(np.nanmean((true_val - ml_pred[val_idx])**2)))
        improvement_threshold = rmse_anchor * 0.98
        if rmse_ml < improvement_threshold:
            tvt_use = ml_pred
            log(f"  ML kept (val RMSE {rmse_ml:.3f} < anchor {rmse_anchor:.3f})")
        else:
            tvt_use = tvt_anchor
            log(f"  ML rejected (val RMSE {rmse_ml:.3f} >= {improvement_threshold:.3f}); using physics anchor")
    elif kn_mask.sum() > 10:
        n_val = max(5, int(kn_mask.sum() * 0.15))
        val_idx = np.where(kn_mask)[0][-n_val:]
        true_val = hw_ref['TVT_input'].values[val_idx]
        rmse_anchor = float(np.sqrt(np.nanmean((true_val - tvt_anchor[val_idx])**2)))
        rmse_ml = float(np.sqrt(np.nanmean((true_val - ml_pred[val_idx])**2)))
        improvement_threshold = rmse_anchor * 0.98
        if rmse_ml < improvement_threshold:
            tvt_use = ml_pred
            log(f"  ML kept (val RMSE {rmse_ml:.3f} < anchor {rmse_anchor:.3f})")
        else:
            tvt_use = tvt_anchor
            log(f"  ML rejected (val RMSE {rmse_ml:.3f} >= {improvement_threshold:.3f}); using physics anchor")
    else:
        tvt_use = tvt_anchor
        log("  Not enough known rows to validate ML correction; using physics anchor")

    # Gold overlay -> guarded contact
    tvt_overlay = gold_overlay_calibrate(hw_ref, tvt_use)
    tvt_final = guarded_contact_override(hw_ref, tw_ref, tvt_overlay)

    if hw_tr is not None:
        km = hw_tr['TVT_input'].notna().values
        tvt_final[km] = hw_tr['TVT_input'].values[km]
    else:
        km = hw_te['TVT_input'].notna().values
        tvt_final[km] = hw_te['TVT_input'].values[km]

    # Optimized Savitzky-Golay filtering for final smoothing
    window = min(17, len(tvt_final)//2*2+1)
    if window >= 5:
        tvt_final = savgol_filter(tvt_final, window_length=window, polyorder=3, mode='interp')
    
    # Final consistency check
    if hw_tr is not None:
        km_idx = np.where(hw_tr['TVT_input'].notna().values)[0]
    else:
        km_idx = np.where(hw_te['TVT_input'].notna().values)[0]
    
    if len(km_idx) > 20:
        tvt_final[km_idx] = savgol_filter(tvt_final[km_idx], 
                                          window_length=min(7, len(km_idx)//2*2+1), 
                                          polyorder=2, mode='interp')

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        rows.append({'id': row['id'], 'tvt': float(tvt_final[ridx])})

    elapsed_well = time.time() - t_well
    times_per_well.append(elapsed_well)
    avg_t = np.mean(times_per_well)
    remaining = (n_wells - wi - 1) * avg_t
    log(f"  Done: {len(ws)} rows | Well: {elapsed_well:.1f}s | ETA: {remaining/60:.1f} min")

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
log(f"\n✅ submission.csv: {len(submission)} rows")
log(f"   TVT: mean={submission['tvt'].mean():.2f}  std={submission['tvt'].std():.2f}")
log(f"   Total: {(time.time()-T0)/60:.1f} min")
print(submission.head(10))